<a href="https://colab.research.google.com/github/look4pritam/Agentic-AI/blob/master/Notebooks/LangChain/RAG-vLLM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Retrieval Augmented Generation (RAG) using LangChain.

### Install Python packages.

In [ ]:
!pip install transformers

In [ ]:
!pip install langchain

In [ ]:
!pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 30.7/30.7 MB 65.6 MB/s eta 0:00:00


In [ ]:
!pip install langchain_community langchain_huggingface

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 40.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 115.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 90.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 59.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 32.1 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
 

In [ ]:
!pip install pypdf

### vLLM

In [ ]:
!pip install vllm

In [ ]:
!pip install openai

- Restart the session.

### Run vLLM server.

In [ ]:
!vllm serve --help

INFO 04-14 11:06:20 [__init__.py:239] Automatically detected platform cuda.
2025-04-14 11:06:22.443267: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1744628782.752483    2563 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1744628782.841871    2563 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-04-14 11:06:23.450260: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
usage: vllm serve [model_tag] [options]



In [ ]:
!nohup vllm serve 'unsloth/Llama-3.2-3B-Instruct'\
  --dtype half \
  --api-key password \
  --gpu-memory-utilization 0.70 \
  --max-model-len 16384 \
  --host 0.0.0.0 \
  --port 8000 \
  --trust-remote-code &

nohup: appending output to 'nohup.out'


In [ ]:
!cat nohup.out

INFO 04-14 11:06:47 [__init__.py:239] Automatically detected platform cuda.
2025-04-14 11:06:48.020268: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1744628808.054885    2712 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1744628808.065284    2712 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-04-14 11:06:48.098113: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
INFO 04-14 11:06:52 [api_server.py:1034] 

In [ ]:
import os

os.environ["OPENAI_API_KEY"] = "password"
os.environ["OPENAI_API_BASE"] = "http://localhost:8000/v1"
os.environ["OPENAI_MODEL_NAME"] = "unsloth/Llama-3.2-3B-Instruct"

In [ ]:
from langchain.chat_models import ChatOpenAI

In [ ]:
large_language_model = ChatOpenAI(
    model="unsloth/Llama-3.2-3B-Instruct",
    base_url="http://localhost:8000/v1",
    api_key="password"
    )

<ipython-input-162-43ce2b62e66a>:1: LangChainDeprecationWarning: The class `ChatOpenAI` was deprecated in LangChain 0.0.10 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-openai package and should be used instead. To use it run `pip install -U :class:`~langchain-openai` and import as `from :class:`~langchain_openai import ChatOpenAI``.
  large_language_model = ChatOpenAI(


### Use 'unsloth/Llama-3.2-3B-Instruct' LLM.

The Llama-3.2-3B-Instruct Large Language Model (LLM) is a instruct fine-tuned version of the Llama-3.2-3B generative text model using a variety of publicly available conversation datasets.

See [link](https://huggingface.co/unsloth/Llama-3.2-3B-Instruct) for more details.

In [ ]:
model_name = 'unsloth/Llama-3.2-3B'

### Create a tokenizer.

In [ ]:
from transformers import AutoTokenizer

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)

### Load pre-trained model.

In [ ]:
from transformers import AutoModelForCausalLM

In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map='auto'
)

### Create 'text-generation' pipeline.

In [ ]:
from transformers import pipeline

In [ ]:
text_generation_pipeline = pipeline(
    model=model,
    tokenizer=tokenizer,
    task="text-generation",
    temperature=0.2,
    repetition_penalty=1.1,
    return_full_text=True,
    max_new_tokens=1000,
)

In [ ]:
from langchain.llms import HuggingFacePipeline

In [ ]:
large_language_model = HuggingFacePipeline(pipeline=text_generation_pipeline)

### Create a document loader.

In [ ]:
from langchain.document_loaders import PyPDFLoader

In [ ]:
loader = PyPDFLoader("document.pdf")
documents = loader.load()

In [ ]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
document_chunks = text_splitter.split_documents(documents)

In [ ]:
for index, document_chunk in enumerate(document_chunks):
    print(f"Chunk {index+1}: {document_chunk.page_content[:200]}...")

Chunk 1: CS Ashwani Singh Bisht, ACS
Company Secretary & Compliance Officer  
Bartronics India Limited  
Madhapur, Hyderabad  
ashwanisingh900@gmail.com
Employee Stock Option Plan (ESOP) : The Finer 
Nuances
A...
Chunk 2: employee benefit plan that gives workers ownership 
interest in the company in the form of shares or stock 
of the company. One thing we should keep in mind that 
it is an option and it is not an obli...
Chunk 3: of ESOP to claim the benefit of the ESOP. This period 
is called vesting period. After the completion of the 
vesting period the employees become eligible to purchase 
the specified number of shares o...
Chunk 4: such scheme by way of passing special resolution 
subject to the conditions specified under Rule 12, 
of Companies (Share Capital and Debentures)  
Rules, 2014.
However, a Specified IFSC Public Compan...
Chunk 5: completed.
Exercise period:  It means the time period which starts 
after the completion of vesting period within which an 
employee can ex

### Create vector database.

In [ ]:
from langchain.vectorstores import FAISS

In [ ]:
from langchain.embeddings.huggingface import HuggingFaceEmbeddings

In [ ]:
db = FAISS.from_documents(document_chunks,
                          HuggingFaceEmbeddings(model_name='sentence-transformers/all-mpnet-base-v2'))

<ipython-input-170-ef68b5ffe52a>:2: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  HuggingFaceEmbeddings(model_name='sentence-transformers/all-mpnet-base-v2'))
/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
 

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.4k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
retriever = db.as_retriever(search_type="similarity",
                            search_k=3)

### Define a tool for an agent.

In [ ]:
from langchain.tools import Tool

In [ ]:
def get_weather(location: str):
    return f"The current weather in {location} is sunny and 25°C."

In [ ]:
weather_tool = Tool(
    name="Weather Tool",
    func=get_weather,
    description="Provides weather information for a given location."
)

### Build a Retrieval Question-Answer chain.

In [ ]:
from langchain.chains import RetrievalQA

In [ ]:
retrieval_qa_chain = RetrievalQA.from_chain_type(
    llm=large_language_model,
    retriever=retriever,
    return_source_documents=True
)

###  Initialize the Agent.

In [ ]:
from langchain.agents import initialize_agent, Tool, AgentType

In [ ]:
tools = [
    Tool(
        name="Document Retrieval",
        func=lambda q: retrieval_qa_chain({"query": q})["result"],
        description="Retrieve knowledge from the document database."
    ),
    weather_tool
]

In [ ]:
agent = initialize_agent(
    tools=tools,
    llm=large_language_model,
    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True
)

<ipython-input-179-e66ea08ba3fc>:1: LangChainDeprecationWarning: LangChain agents will continue to be supported, but it is recommended for new use cases to be built with LangGraph. LangGraph offers a more flexible and full-featured framework for building agents, including support for tool-calling, persistence of state, and human-in-the-loop workflows. For details, refer to the `LangGraph documentation <https://langchain-ai.github.io/langgraph/>`_ as well as guides for `Migrating from AgentExecutor <https://python.langchain.com/docs/how_to/migrate_agent/>`_ and LangGraph's `Pre-built ReAct agent <https://langchain-ai.github.io/langgraph/how-tos/create-react-agent/>`_.
  agent = initialize_agent(


### Sample queries.

In [ ]:
user_query = "What is Employee Stock Option Plan (ESOP) ?"
response = agent.invoke(user_query)
print('response', response)



> Entering new AgentExecutor chain...
Thought: I need to retrieve information about Employee Stock Option Plan (ESOP) from the document database.

Action: Document Retrieval
Action Input: ESOP


<ipython-input-178-d627a05d4e53>:4: LangChainDeprecationWarning: The method `Chain.__call__` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use :meth:`~invoke` instead.
  func=lambda q: retrieval_qa_chain({"query": q})["result"],



Observation: An Employee Stock Option Plan (ESOP) is a type of employee benefit plan that gives employees the right to purchase shares of their employer company at a pre-determined price after a certain time period.

Here are some key points about ESOPs:

1. **Definition**: ESOP stands for Employee Stock Option Plan.
2. **Purpose**: To give employees ownership interest in the company.
3. **How it works**: Employees are granted options to buy shares at a discounted price, which is usually lower than the market price.
4. **Eligibility**: Employees who are eligible for ESOPs are typically those who are exclusively working for the company or its subsidiaries, and are not part of the promoter group or holding more than 10% of the company's paid-up share capital.
5. **Supplements salary**: ESOPs are often used to supplement employee salaries, rather than replace them.
6. **Risk**: Accepting ESOPs instead of a higher salary can be risky if the company's performance is not good, as the employ

In [ ]:
user_query = "What's the weather in Mumbai ?"
response = agent.run(user_query)
print('response', response)



> Entering new AgentExecutor chain...
Thought: To find the weather in Mumbai, I should use the Weather Tool.

Action: Weather Tool
Action Input: Mumbai

Observation: The current weather in Mumbai
 is sunny and 25°C.
Thought:Question: What's the weather in Mumbai ?
Thought: Thought: To find the weather in Mumbai, I should use the Weather Tool.

Action: Weather Tool
Action Input: Mumbai

Observation: The current weather in Mumbai
 is sunny and 25°C.
Thought:Question: What's the weather in Mumbai ?
Thought: To find the weather in Mumbai, I should use the Weather Tool.

Action: Weather Tool
Action Input: Mumbai

Observation: The current weather in Mumbai
 is sunny and 25°C.
Thought:Question: What's the weather in Mumbai ?
Thought: To find the weather in Mumbai, I should use the Weather Tool.

Action: Weather Tool
Action Input: Mumbai

Observation: The current weather in Mumbai
 is sunny and 25°C.
Thought:Question: What's the weather in Mumbai ?
Thought: To find the weather in Mumbai, I s

In [ ]:
user_query = "What is Employee Stock Option Plan (ESOP) ? and tell me the weather in Pune."
response = agent.run(user_query)
print('response', response)



> Entering new AgentExecutor chain...
Question: What is Employee Stock Option Plan (ESOP) ? and tell me the weather in Pune.
Thought: I need to retrieve information about Employee Stock Option Plan (ESOP) from the document database.

Action: Document Retrieval
Action Input: ESOP
Observation: An ESOP stands for Employee Stock Option Plan. It is a type of employee benefit plan that gives employees the right to purchase shares of their employer company at a pre-determined price after a certain time period.

ESOPs are often used to supplement the salaries of employees, as they may generate more wealth for employees if the company is growing and generating good earnings above the break-even point.

However, ESOPs can be risky if an employee accepts them instead of a higher salary and the company's performance is not as expected. In such cases, the employee may end up with a loss.

To be eligible for an ESOP, an employee must meet certain criteria, such as being exclusively employed by the